# Probabilistic Matrix Factorization (PMF) con Neural Collaborative Filtering

Este notebook implementa un modelo PMF puro. A diferencia de GMF (que aprende una capa lineal de pesos sobre el producto Hadamard), PMF basa su predicción estrictamente en el producto escalar de los factores latentes:

$$\hat{r}_{u,i} = \sigma(\mathbf{p}_u^T \mathbf{q}_i) \cdot 4.0 + 1.0$$

**Metodología Estricta:**
1. **Train**: Ajuste de pesos (85% del dataset de entrenamiento original).
2. **Validación**: Ajuste de hiperparámetros y *Early Stopping* (15% del dataset de entrenamiento original).
3. **Test**: Evaluación final intocable con *Negative Sampling*. Cero exposición durante el entrenamiento.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import time, json, sys, os, random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Trabajo_Musica_CSP/'
PATH_DATA = BASE + 'Data/'

# 1. Carga de datos
df_train = pd.read_csv(PATH_DATA + 'train.csv')
df_test = pd.read_csv(PATH_DATA + 'test.csv')

num_users = df_train['user_n'].max() + 1
num_items = df_train['track_n'].max() + 1

# 2. Split metodológicamente correcto (Train puro vs Validación)
df_train_puro, df_val = train_test_split(df_train, test_size=0.15, random_state=42)

# 3. Creación de Tensores
user_train = torch.LongTensor(df_train_puro['user_n'].values).to(device)
item_train = torch.LongTensor(df_train_puro['track_n'].values).to(device)
rating_train = torch.FloatTensor(df_train_puro['rating'].values).to(device)
td_train = TensorDataset(user_train, item_train, rating_train)

user_val = torch.LongTensor(df_val['user_n'].values).to(device)
item_val = torch.LongTensor(df_val['track_n'].values).to(device)
rating_val = torch.FloatTensor(df_val['rating'].values).to(device)
td_val = TensorDataset(user_val, item_val, rating_val)

user_test = torch.LongTensor(df_test['user_n'].values).to(device)
item_test = torch.LongTensor(df_test['track_n'].values).to(device)
rating_test = torch.FloatTensor(df_test['rating'].values).to(device)
td_test = TensorDataset(user_test, item_test, rating_test)

# DataLoaders
dl_train = DataLoader(td_train, batch_size=1024, shuffle=True)
dl_val = DataLoader(td_val, batch_size=1024, shuffle=False)
dl_test = DataLoader(td_test, batch_size=1024, shuffle=False)

print("Datos listos. Train:", len(td_train), "| Val:", len(td_val), "| Test:", len(td_test))

## 2. Definición del Modelo PMF y Funciones Maestras

In [ ]:
class PMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, latent_dim)
        self.item_emb = nn.Embedding(num_items, latent_dim)
        
        # Inicialización normal para estabilizar el producto escalar temprano
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)
        i = self.item_emb(item_ids)
        
        # Producto escalar: multiplicar elemento a elemento y sumar en la dimensión latente
        dot_product = (u * i).sum(dim=1)
        
        # Acotar predicción [1, 5] usando sigmoide
        pred = torch.sigmoid(dot_product) * 4.0 + 1.0
        return pred

def evaluar_modelo(modelo, dataloader, loss_fn):
    modelo.eval()
    loss_total, n_muestras = 0, 0
    with torch.no_grad():
        for users, items, ratings in dataloader:
            preds = modelo(users, items)
            loss = loss_fn(preds, ratings)
            loss_total += loss.item() * len(ratings)
            n_muestras += len(ratings)
    return loss_total / n_muestras

def entrenar_modelo(modelo, train_loader, val_loader, lr, num_epocas, paciencia=3, verbose=True):
    optimizer = torch.optim.Adam(modelo.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    historial_train, historial_val = [], []
    mejor_val_loss = float('inf')
    mejor_epoca = 0
    mejor_modelo_state = None
    epocas_sin_mejora = 0

    for epoch in range(num_epocas):
        start = time.time()
        # Entrenar
        modelo.train()
        loss_train_acum = 0
        for users, items, ratings in train_loader:
            optimizer.zero_grad()
            preds = modelo(users, items)
            loss = loss_fn(preds, ratings)
            loss.backward()
            optimizer.step()
            loss_train_acum += loss.item() * len(ratings)
            
        loss_train = loss_train_acum / len(train_loader.dataset)
        
        # Validar
        loss_val = evaluar_modelo(modelo, val_loader, loss_fn)
        historial_train.append(loss_train)
        historial_val.append(loss_val)

        # Early Stopping
        if loss_val < mejor_val_loss - 1e-4:
            mejor_val_loss = loss_val
            mejor_epoca = epoch + 1
            mejor_modelo_state = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
            epocas_sin_mejora = 0
            marker = " > (Mejora)"
        else:
            epocas_sin_mejora += 1
            marker = " <"

        if verbose:
            print(f"Epoca {epoch+1} | train: {loss_train:.4f} | val: {loss_val:.4f} ({time.time()-start:.1f}s){marker}")

        if epocas_sin_mejora >= paciencia:
            if verbose: print(f"  Early stopping activado. Mejor modelo: época {mejor_epoca}")
            break

    if mejor_modelo_state is not None:
        modelo.load_state_dict(mejor_modelo_state)

    return modelo, historial_train, historial_val, mejor_epoca, mejor_val_loss

## 3. Grid Search de Hiperparámetros (Monitorizado en Validación)

In [ ]:
resultados_path_pmf = BASE + 'pmf_grid_results.json'

pruebas = [
    (8, 0.005),  (8, 0.01),  (8, 0.02),
    (16, 0.005), (16, 0.01), (16, 0.02),
    (32, 0.005), (32, 0.01), (32, 0.02)
]
epocas = 10

try:
    with open(resultados_path_pmf, 'r') as f:
        results = json.load(f)
    results = {tuple(map(float, k.split(','))): v for k, v in results.items()}
    print(f"Resultados previos cargados: {len(results)} configs ya hechas")
except FileNotFoundError:
    results = {}
    print("Empezando Grid Search desde cero")

for i, (ld, lr) in enumerate(pruebas):
    config = (float(ld), float(lr))
    if config in results: 
        print(f"[{i+1}/{len(pruebas)}] dim={int(ld)}, lr={lr}: YA HECHO")
        continue

    print(f"\n[{i+1}/{len(pruebas)}] Entrenando PMF dim={int(ld)}, lr={lr}...")
    start = time.time()

    modelo_grid = PMF(num_users, num_items, latent_dim=int(ld)).to(device)
    
    _, hist_train, hist_val, mejor_ep, mejor_val_loss = entrenar_modelo(
        modelo_grid, dl_train, dl_val, lr=lr, num_epocas=epocas, paciencia=3, verbose=False
    )

    results[config] = {
        'hist_train': hist_train, 'hist_val': hist_val,
        'loss_val_minimo': mejor_val_loss, 'mejor_epoca': mejor_ep, 'segundos': time.time() - start
    }
    
    resultado_str = {f"{k[0]},{k[1]}": v for k, v in results.items()}
    with open(resultados_path_pmf, 'w') as f:
        json.dump(resultado_str, f, indent=2)
    print(f"  Mínima Val Loss: {mejor_val_loss:.4f} (época {mejor_ep})")

## 4. Análisis de Generalización (Extracción sin Data Leakage)
Extraemos las curvas de validación registradas durante el Grid Search para determinar la configuración óptima sin contaminar el conjunto de test.

In [ ]:
lr_objetivo = 0.01

try:
    hist_val_8 = results[(8.0, lr_objetivo)]['hist_val']
    hist_val_16 = results[(16.0, lr_objetivo)]['hist_val']
    hist_val_32 = results[(32.0, lr_objetivo)]['hist_val']
    
    plt.figure(figsize=(12, 6))
    plt.plot(range(1, len(hist_val_32) + 1), hist_val_32, marker='s', color='#d62728', label='PMF dim=32')
    plt.plot(range(1, len(hist_val_16) + 1), hist_val_16, marker='^', color='#1f77b4', label='PMF dim=16')
    plt.plot(range(1, len(hist_val_8) + 1), hist_val_8, marker='o', color='#2ca02c', label='PMF dim=8')

    plt.xlabel("Época")
    plt.ylabel("Loss en VALIDACIÓN (MSE)")
    plt.title(f"Comparativa PMF: Generalización según dimensión latente (lr={lr_objetivo})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
except KeyError:
    print("Faltan datos en el JSON para graficar.")

# Obtener los hiperparámetros ganadores globales
mejor = min(results.items(), key=lambda x: x[1]['loss_val_minimo'])
MEJOR_DIM_PMF = int(mejor[0][0])
MEJOR_LR_PMF = mejor[0][1]

print("\n" + "="*50)
print(f"CONFIGURACIÓN ÓPTIMA PMF GANADORA:")
print(f"-> Latent Dim: {MEJOR_DIM_PMF}")
print(f"-> Learning Rate: {MEJOR_LR_PMF}")
print(f"-> Loss Val Mínima: {mejor[1]['loss_val_minimo']:.4f}")
print("="*50)

## 5. Entrenamiento del Modelo Final y Evaluación Rigurosa en Test
Se entrena un único modelo con los parámetros ganadores, congelándolo en su punto de máxima generalización mediante validación cruzada. Finalmente, se evalúa contra Test usando muestreo negativo (1 positivo + 99 negativos).

In [ ]:
sys.path.append(BASE)
from metrics import evaluate_model

final_pmf_path = BASE + 'pmf_final_model.pth'
metrics_path_pmf = BASE + 'pmf_metrics.json'
preds_path_pmf = BASE + 'pmf_predictions.csv'

modelo_final_pmf = PMF(num_users, num_items, latent_dim=MEJOR_DIM_PMF).to(device)
loss_fn = nn.MSELoss()

# 1. ENTRENAMIENTO DEFINITIVO
if os.path.exists(final_pmf_path):
    print("Cargando modelo final óptimo PMF desde Drive...")
    modelo_final_pmf.load_state_dict(torch.load(final_pmf_path))
else:
    print(f"Entrenando modelo definitivo (dim={MEJOR_DIM_PMF}, lr={MEJOR_LR_PMF}) con Early Stopping...")
    optimizer = torch.optim.Adam(modelo_final_pmf.parameters(), lr=MEJOR_LR_PMF)
    mejor_loss_val = float('inf')
    paciencia, epocas_sin_mejora = 3, 0
    
    for epoch in range(15):
        modelo_final_pmf.train()
        for u, i, r in dl_train:
            optimizer.zero_grad()
            loss = loss_fn(modelo_final_pmf(u, i), r)
            loss.backward()
            optimizer.step()
            
        loss_val = evaluar_modelo(modelo_final_pmf, dl_val, loss_fn)
        print(f"Época {epoch+1} | Val Loss: {loss_val:.4f}")
        
        if loss_val < mejor_loss_val - 1e-4:
            mejor_loss_val = loss_val
            torch.save(modelo_final_pmf.state_dict(), final_pmf_path)
            epocas_sin_mejora = 0
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= paciencia:
                print("Early stopping activado.")
                break
                
    modelo_final_pmf.load_state_dict(torch.load(final_pmf_path))

# 2. EVALUACIÓN REGRESIÓN (TEST)
loss_test = evaluar_modelo(modelo_final_pmf, dl_test, loss_fn)
print(f"\n--- RENDIMIENTO REGRESIÓN (TEST) ---")
print(f"MSE:  {loss_test:.4f}")
print(f"RMSE: {loss_test ** 0.5:.4f}\n")

# 3. EVALUACIÓN RANKING (NEGATIVE SAMPLING)
if os.path.exists(metrics_path_pmf):
    with open(metrics_path_pmf, 'r') as f:
        resultados = json.load(f)
    print("Métricas de ranking cargadas desde Drive.")
else:
    print("Preparando Negative Sampling (1 Positivo + 99 Negativos)...")
    interacciones_train = df_train.groupby('user_n')['track_n'].apply(set).to_dict()
    eval_users, eval_items, eval_ratings = [], [], []

    for _, row in df_test.iterrows():
        u, i_real, r_real = int(row['user_n']), int(row['track_n']), float(row['rating'])
        eval_users.append(u)
        eval_items.append(i_real)
        eval_ratings.append(r_real)

        vistos = interacciones_train.get(u, set())
        vistos.add(i_real)
        negativos = []
        while len(negativos) < 99:
            neg_item = random.randint(0, num_items - 1)
            if neg_item not in vistos:
                negativos.append(neg_item)
                vistos.add(neg_item)

        eval_users.extend([u] * 99)
        eval_items.extend(negativos)
        eval_ratings.extend([1.0] * 99)

    # Inferencia por batches
    modelo_final_pmf.eval()
    preds_test_list = []
    tensor_u = torch.LongTensor(eval_users).to(device)
    tensor_i = torch.LongTensor(eval_items).to(device)
    
    with torch.no_grad():
        for idx in range(0, len(eval_users), 4096):
            users_batch = tensor_u[idx:idx + 4096]
            items_batch = tensor_i[idx:idx + 4096]
            preds_batch = modelo_final_pmf(users_batch, items_batch)
            preds_test_list.append(preds_batch.cpu().numpy())

    preds_test = np.concatenate(preds_test_list)
    df_eval = pd.DataFrame({'user_n': eval_users, 'track_n': eval_items, 'rating': eval_ratings, 'prediction': preds_test})

    print("Calculando métricas oficiales...")
    resultados = evaluate_model(df_eval[['user_n', 'rating']], preds_test, k=10, threshold=4.0)
    
    df_eval.to_csv(preds_path_pmf, index=False)
    with open(metrics_path_pmf, 'w') as f:
        json.dump(resultados, f, indent=2)

print("=" * 50)
print("RESULTADOS RANKING PMF (Negative Sampling)")
print("=" * 50)
for k, v in resultados.items():
    print(f"  {k:15s}: {v:.4f}")